# Chapter 11: Testing and Validation
## AI for Networking and Security Engineers — Volume 1

This notebook demonstrates all testing layers from Chapter 11:

1. **Why AI Testing Is Different** — determinism vs non-determinism comparison
2. **Prompt Unit Tests** — temperature=0 and keyword assertion
3. **Regression Baseline Testing** — detect prompt drift and model changes
4. **LLM-as-a-Judge** — use Sonnet to evaluate Haiku's quality
5. **Semantic Similarity** — test meaning, not exact words
6. **Shadow Deployment** — safe model rollout simulation
7. **Full Quality Gate** — all layers running as a CI/CD pipeline

> **Note**: Replace `YOUR_API_KEY` with your real Anthropic API key to run live AI demos.

In [ ]:
!pip install anthropic -q
print('Dependencies installed.')

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab secrets.')
except Exception:
    import getpass
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter Anthropic API key: ')
    print('API key set via input.')

---
## Demo 1: Why AI Testing Is Different

Traditional software is deterministic. AI is probabilistic.
The same prompt can produce different outputs at different temperatures.

This demo proves that temperature=0 gives much more consistent outputs — the key to testable AI.

In [ ]:
from anthropic import Anthropic
import time

client = Anthropic()

def call_model(prompt: str, temperature: float = 0, model: str = "claude-haiku-4-5-20251001") -> str:
    resp = client.messages.create(
        model=model,
        max_tokens=100,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text.strip()


test_prompt = """Classify this syslog: %OSPF-5-ADJCHG: Neighbor Down
Return ONLY: CRITICAL, HIGH, WARNING, or INFO"""

print("Determinism comparison: 5 calls, same prompt")
print("=" * 55)

print("\nWith temperature=0 (testable):")
t0_results = [call_model(test_prompt, temperature=0) for _ in range(5)]
for i, r in enumerate(t0_results):
    print(f"  Run {i+1}: {r}")
unique_t0 = len(set(t0_results))
print(f"  Unique outputs: {unique_t0}/5 ({'✅ Deterministic' if unique_t0 == 1 else '⚠️ Some variation'})")

print("\nWith temperature=0.9 (creative, hard to test):")
t09_results = [call_model(test_prompt, temperature=0.9) for _ in range(5)]
for i, r in enumerate(t09_results):
    print(f"  Run {i+1}: {r}")
unique_t09 = len(set(t09_results))
print(f"  Unique outputs: {unique_t09}/5")

print("\n" + "="*55)
print("Rule: Always use temperature=0 in test code.")
print("Save creative temperatures for generation tasks.")

### The Key Difference: AI Is Non-Deterministic

Traditional software: same input → same output, every time.
AI models: same input → *similar* output, but never byte-identical.

This means you cannot test AI with `assert output == expected_string`.
Instead, you test for **semantic correctness**:

- Does the answer contain the right keywords?
- Is the JSON schema valid?
- Does a judge model score it above 7/10?
- Is it similar enough to the baseline (>80% match)?

> **Network analogy:** You don't test a router by checking if the exact same bytes
> come out. You test if the packet reaches the right destination with the right TTL.
> Same principle: test the *meaning*, not the exact characters.

---
## Demo 2: Prompt Unit Tests — The Telnet Test

This is the test that would have prevented the telnet incident.
Run this in CI and the 'minor prompt simplification' never reaches production.

In [ ]:
from anthropic import Anthropic
import json, re

client = Anthropic()


def call_model(prompt: str, max_tokens: int = 500) -> str:
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text


def assert_contains(text: str, keywords: list) -> bool:
    text_lower = text.lower()
    found = [kw for kw in keywords if kw.lower() in text_lower]
    return len(found) > 0, found


# Two prompt versions: good vs bad
PROMPT_GOOD = """Analyze this Cisco IOS config for security issues.
Flag ALL problems including weak authentication, insecure protocols, and weak SNMP.
Config: {config}
Return JSON: {{"severity": "low|medium|high|critical", "issues": ["list"]}}"""

PROMPT_BAD = """Analyze for security issues: {config}"""  # The 'simplified' version

test_cases = [
    {
        "name": "Telnet Detection (THE telnet incident test)",
        "config": "line vty 0 4\n transport input telnet\n no login",
        "must_find": ["telnet", "ssh"]
    },
    {
        "name": "Weak SNMP",
        "config": "snmp-server community public RO\nsnmp-server community private RW",
        "must_find": ["snmp", "community", "public"]
    },
    {
        "name": "No password encryption",
        "config": "no service password-encryption\nenable password cisco",
        "must_find": ["password", "encrypt"]
    }
]

print("Prompt Unit Test Results")
print("=" * 60)

for prompt_label, prompt_template in [("GOOD (detailed)", PROMPT_GOOD), ("BAD (simplified)", PROMPT_BAD)]:
    print(f"\nPrompt version: {prompt_label}")
    print("-" * 40)
    passed = 0
    total = len(test_cases)

    for tc in test_cases:
        prompt = prompt_template.format(config=tc["config"])
        response = call_model(prompt)
        ok, found = assert_contains(response, tc["must_find"])
        icon = "✅" if ok else "❌"
        print(f"  {icon} {tc['name']}")
        if not ok:
            print(f"     Expected: {tc['must_find']}")
            print(f"     Response: {response[:150]}")
        else:
            passed += 1

    print(f"  Score: {passed}/{total} {'✅ PASS' if passed == total else '❌ FAIL — would block this commit in CI'}")

print("\n" + "="*60)
print("The bad prompt would FAIL the CI test.")
print("The production push would be blocked. Telnet incident prevented.")

---
## Demo 3: Regression Baseline Testing

On first run: establishes a baseline (saves to JSON file).
On subsequent runs: compares against baseline, fails if behavior changed.

This catches model drift when Anthropic releases a new Haiku version.

In [ ]:
from anthropic import Anthropic
import json, re, hashlib, time, os

client = Anthropic()

# Use /tmp for Colab
BASELINE_FILE = "/tmp/ai_test_baselines.json"


def load_baselines() -> dict:
    if os.path.exists(BASELINE_FILE):
        with open(BASELINE_FILE) as f:
            return json.load(f)
    return {}


def save_baselines(b: dict):
    with open(BASELINE_FILE, "w") as f:
        json.dump(b, f, indent=2)


def run_regression_test(
    test_id: str,
    prompt: str,
    expected_keywords: list,
    force_baseline: bool = False
) -> dict:
    baselines = load_baselines()

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    text = resp.content[0].text.lower()

    found = [kw for kw in expected_keywords if kw.lower() in text]
    missing = [kw for kw in expected_keywords if kw.lower() not in text]
    coverage = len(found) / len(expected_keywords) * 100

    current = {
        "keywords_found": found,
        "keywords_missing": missing,
        "coverage_pct": coverage,
        "prompt_hash": hashlib.md5(prompt.encode()).hexdigest()[:8],
        "timestamp": time.strftime("%Y-%m-%d %H:%M")
    }

    if test_id not in baselines or force_baseline:
        baselines[test_id] = current
        save_baselines(baselines)
        return {"status": "baseline_created", "coverage": coverage, "found": found}

    baseline = baselines[test_id]
    lost = set(baseline["keywords_found"]) - set(found)
    coverage_drop = baseline["coverage_pct"] - coverage

    regressions = []
    if lost:
        regressions.append(f"Lost keywords: {lost}")
    if coverage_drop > 15:
        regressions.append(f"Coverage dropped {coverage_drop:.0f}%")

    return {
        "status": "regression_detected" if regressions else "passed",
        "coverage": coverage,
        "baseline_coverage": baseline["coverage_pct"],
        "regressions": regressions,
        "found": found
    }


test_suite = [
    {
        "id": "security_telnet_snmp",
        "prompt": """Security analysis for Cisco IOS:
line vty 0 4\n transport input telnet\nsnmp-server community public RO\nReturn JSON: {"issues": []}""",
        "keywords": ["telnet", "snmp", "community", "ssh"]
    },
    {
        "id": "bgp_down_classification",
        "prompt": "%BGP-5-ADJCHANGE: neighbor 10.1.1.2 Down — classify severity",
        "keywords": ["critical", "bgp", "neighbor"]
    },
    {
        "id": "interface_extraction",
        "prompt": "Extract: interface GigabitEthernet0/0\n ip address 10.1.1.1 255.255.255.252\nJSON: {name, ip}",
        "keywords": ["gigabit", "10.1.1.1"]
    }
]

print("Regression Test Suite (first run creates baselines)")
print("=" * 55)

# First run
print("\nRun 1 (baseline creation):")
for tc in test_suite:
    result = run_regression_test(tc["id"], tc["prompt"], tc["keywords"], force_baseline=True)
    print(f"  📋 {tc['id']}: baseline set ({result['coverage']:.0f}% coverage, found: {result['found']})")

# Second run (should pass)
print("\nRun 2 (regression check, same prompts — should pass):")
for tc in test_suite:
    result = run_regression_test(tc["id"], tc["prompt"], tc["keywords"])
    icon = "✅" if result["status"] == "passed" else "❌"
    print(f"  {icon} {tc['id']}: {result['status']} ({result['coverage']:.0f}% vs baseline {result['baseline_coverage']:.0f}%)")

# Simulate a broken prompt (would trigger regression)
print("\nRun 3 (simulating weakened prompt — should detect regression):")
broken_result = run_regression_test(
    "security_telnet_snmp",
    "Analyze: line vty 0 4\n transport input telnet",  # Shorter, less specific
    ["telnet", "snmp", "community", "ssh"]
)
icon = "✅" if broken_result["status"] == "passed" else "⚠️"
print(f"  {icon} security_telnet_snmp: {broken_result['status']}")
if broken_result.get("regressions"):
    for r in broken_result["regressions"]:
        print(f"     Regression: {r}")

print("\n✓ Regression testing demo complete!")

---
## Demo 4: LLM-as-a-Judge — Quality Evaluation with Rubrics

Use Sonnet to evaluate Haiku's config analysis quality.
This catches hallucinations, wrong remediation commands, and missed issues
that keyword tests would miss.

Score: 5=Excellent, 4=Good, 3=Acceptable, 2=Poor, 1=Fail

In [ ]:
from anthropic import Anthropic
import json, re

client = Anthropic()


def get_ai_analysis(config: str) -> str:
    """Student model — Haiku analyzes the config."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=800,
        temperature=0,
        messages=[{"role": "user", "content": f"""
Analyze this Cisco IOS config for security issues.
Config:
```
{config}
```
Return JSON: {{"severity": "low|medium|high|critical", "issues": [{{"description": "...", "remediation": "exact CLI"}}]}}"""}]
    )
    return resp.content[0].text


def judge_analysis(config: str, ai_output: str, ground_truth: list) -> dict:
    """Judge model — Sonnet evaluates Haiku's output quality."""
    resp = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        temperature=0,
        messages=[{"role": "user", "content": f"""
You are a senior network security engineer evaluating an AI assistant's analysis.

Scoring: 5=All issues found, correct remediation, no false positives
         4=Main issues found, minor gaps OK
         3=Core issues found, some missed
         2=Multiple critical issues missed
         1=Failed to catch obvious issues

CONFIG ANALYZED:
```
{config}
```

AI RESPONSE:
```
{ai_output}
```

GROUND TRUTH ISSUES:
{json.dumps(ground_truth, indent=2)}

Return JSON: {{"score": 1-5, "issues_found": [], "issues_missed": [], "false_positives": [], "explanation": "..."}}"""}]
    )
    text = resp.content[0].text
    m = re.search(r'\{.*\}', text, re.DOTALL)
    return json.loads(m.group()) if m else {"score": 0, "explanation": "Parse failed"}


test_cases = [
    {
        "name": "Multi-issue router config",
        "config": """
hostname rtr-01
line vty 0 4
 login
 transport input telnet
!
snmp-server community public RO
snmp-server community private RW
!
no service password-encryption
enable password cisco
""",
        "ground_truth": [
            "No authentication on VTY (login without local/aaa)",
            "Telnet enabled on VTY lines",
            "Default SNMP community 'public' (RO)",
            "Default SNMP community 'private' (RW)",
            "Password encryption disabled",
            "Weak enable password 'cisco'"
        ]
    },
    {
        "name": "Mostly secure config",
        "config": """
hostname rtr-02
service password-encryption
!
ip ssh version 2
!
line vty 0 4
 login local
 transport input ssh
 access-class VTY-ACL in
!
snmp-server community xK9#mQ_secret RO 15
""",
        "ground_truth": [
            "No maximum-prefix protection on SSH sessions (minor)"
        ]
    }
]

print("LLM-as-a-Judge Evaluation")
print("=" * 55)
total_score = 0

for tc in test_cases:
    print(f"\nTest: {tc['name']}")
    print("-" * 40)

    # Haiku analyzes the config
    ai_output = get_ai_analysis(tc["config"])
    print(f"Haiku output (first 200): {ai_output[:200]}...")

    # Sonnet judges Haiku's output
    evaluation = judge_analysis(tc["config"], ai_output, tc["ground_truth"])
    score = evaluation.get("score", 0)
    total_score += score

    score_icon = "✅" if score >= 4 else "⚠️" if score == 3 else "❌"
    print(f"\n{score_icon} Judge Score: {score}/5")
    print(f"Issues found : {evaluation.get('issues_found', [])}")
    print(f"Issues missed: {evaluation.get('issues_missed', [])}")
    print(f"False pos    : {evaluation.get('false_positives', [])}")
    print(f"Explanation  : {evaluation.get('explanation', 'N/A')[:300]}")

avg = total_score / len(test_cases)
print(f"\n{'='*55}")
print(f"Average score: {avg:.1f}/5.0")
print(f"Quality gate (>=4.0): {'✅ PASSED' if avg >= 4.0 else '❌ FAILED — fix prompts before deploy'}")
print("\n✓ LLM-as-a-Judge demo complete!")

---
## Demo 5: Semantic Similarity Testing

Keyword tests fail on synonyms. LLM-as-a-Judge is expensive.
Semantic similarity sits in between — tests whether two texts MEAN the same thing.

'Enable SSH instead of telnet' should match 'Use SSH, disable transport input telnet'
even though the exact words are different.

In [ ]:
# Semantic similarity using character n-grams (simplified demo)
# In production: use Voyage AI embeddings via Anthropic SDK for proper semantic vectors

import math


def get_ngram_vector(text: str, n: int = 2) -> dict:
    """Character n-gram frequency vector (simplified embedding)."""
    text = text.lower()
    ngrams = {}
    for i in range(len(text) - n + 1):
        gram = text[i:i+n]
        ngrams[gram] = ngrams.get(gram, 0) + 1
    # Normalize
    total = math.sqrt(sum(v**2 for v in ngrams.values()))
    return {k: v/total for k, v in ngrams.items()} if total > 0 else ngrams


def cosine_similarity(v1: dict, v2: dict) -> float:
    return sum(v1.get(k, 0) * v2.get(k, 0) for k in v1)


def semantic_test(text1: str, text2: str, threshold: float, label: str = "") -> tuple:
    v1 = get_ngram_vector(text1)
    v2 = get_ngram_vector(text2)
    sim = cosine_similarity(v1, v2)
    passed = sim >= threshold
    return sim, passed


print("Semantic Similarity Test Suite")
print("=" * 60)

# Pairs that SHOULD be similar (same concept, different words)
similar_pairs = [
    (
        "Enable SSH and disable telnet for secure remote access",
        "Use SSH version 2 instead of telnet. Configure transport input ssh on VTY lines.",
        0.5, "SSH recommendation"
    ),
    (
        "BGP neighbor session down due to authentication password mismatch",
        "BGP adjacency failure: MD5 authentication key does not match peer configuration",
        0.5, "BGP auth failure"
    ),
    (
        "SNMP community strings public and private must be changed immediately",
        "Replace default SNMP community names. Public/private are known defaults and a security risk.",
        0.5, "SNMP community"
    ),
    (
        "Interface is administratively down. No shutdown command needed.",
        "Interface shutdown: run no shutdown to bring it up",
        0.5, "Interface down"
    ),
]

# Pairs that should be DIFFERENT
different_pairs = [
    (
        "BGP session established successfully. All prefixes received.",
        "Interface GigabitEthernet0/0 is down due to no cable connected.",
        0.7, "BGP vs interface issue"
    ),
]

print("\nPairs that should be semantically similar:")
passed_count = 0
for text1, text2, threshold, label in similar_pairs:
    sim, passed = semantic_test(text1, text2, threshold, label)
    icon = "✅" if passed else "❌"
    print(f"  {icon} [{label}]")
    print(f"     Similarity: {sim:.3f} (threshold: {threshold})")
    print(f"     Text 1: {text1[:60]}...")
    print(f"     Text 2: {text2[:60]}...")
    if passed:
        passed_count += 1

print("\nPairs that should be semantically DIFFERENT:")
for text1, text2, max_threshold, label in different_pairs:
    sim, _ = semantic_test(text1, text2, 0)
    is_different = sim < max_threshold
    icon = "✅" if is_different else "❌"
    print(f"  {icon} [{label}] similarity: {sim:.3f} (should be < {max_threshold})")

print(f"\n{passed_count}/{len(similar_pairs)} similar pairs passed threshold")
print("\nNote: For production use, replace character n-grams with")
print("Voyage AI embeddings via Anthropic SDK for true semantic similarity.")
print("\n✓ Semantic similarity demo complete!")

---
## Demo 6: Shadow Mode Deployment

Safe model rollout: run new model in parallel with production, compare outputs.
Users never see the shadow model's responses — only the production model's.

If divergence rate < 5%, promote the new model.

In [ ]:
from anthropic import Anthropic
import threading, time, json

client = Anthropic()
comparison_log = []
lock = threading.Lock()


def call_model(model: str, prompt: str) -> str:
    resp = client.messages.create(
        model=model,
        max_tokens=200,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text


def check_divergence(output1: str, output2: str) -> bool:
    """Check if two outputs disagree on key severity/classification terms."""
    check_terms = ["critical", "high", "safe", "pass", "fail", "warning"]
    t1 = set(t for t in check_terms if t in output1.lower())
    t2 = set(t for t in check_terms if t in output2.lower())
    return bool(t1 ^ t2)  # Symmetric difference = disagreement


def shadow_call(prompt: str, request_id: str, prod_output: str):
    """Background thread: run shadow model, log comparison."""
    try:
        # In real use: shadow would be a newer model version
        shadow_output = call_model("claude-haiku-4-5-20251001", prompt)
        diverged = check_divergence(prod_output, shadow_output)

        with lock:
            comparison_log.append({
                "request_id": request_id,
                "prod_snippet": prod_output[:100],
                "shadow_snippet": shadow_output[:100],
                "diverged": diverged
            })
    except Exception as e:
        print(f"  Shadow call failed: {e}")


def route_with_shadow(prompt: str, request_id: str) -> str:
    """Route request: always return production, run shadow in background."""
    prod_output = call_model("claude-haiku-4-5-20251001", prompt)

    t = threading.Thread(target=shadow_call, args=(prompt, request_id, prod_output))
    t.daemon = True
    t.start()

    return prod_output  # User only sees this


# Test prompts
prompts = [
    "Analyze: snmp-server community public RO — security issue?",
    "Classify: %BGP-5-ADJCHANGE: neighbor 10.1.1.2 Down",
    "Is this safe? line vty 0 4\n transport input all",
    "Check: interface Gi0/0\n no ip address — problem?",
    "Analyze: ip ssh version 2\n service password-encryption — good practice?",
]

print("Shadow Deployment Demo")
print("=" * 55)
print("Production: haiku-4-5-20251001 (serves users)")
print("Shadow: haiku-4-5-20251001 (same, for demo)")
print("Users see production output only.\n")

for i, prompt in enumerate(prompts):
    result = route_with_shadow(prompt, f"req_{i+1}")
    print(f"Request {i+1}: {prompt[:55]}...")
    print(f"  → Served to user: {result[:80]}...")

time.sleep(2)  # Wait for shadow threads

print("\n" + "="*55)
total = len(comparison_log)
diverged = sum(1 for r in comparison_log if r["diverged"])
divergence_rate = diverged / total if total > 0 else 0

print(f"SHADOW DEPLOYMENT REPORT:")
print(f"  Total compared : {total}")
print(f"  Diverged       : {diverged}")
print(f"  Divergence rate: {divergence_rate:.1%}")
print(f"  Recommendation : {'✅ PROMOTE new model' if divergence_rate < 0.05 else '⚠️ HOLD — review diverged requests'}")

if diverged > 0:
    print("\nDiverged requests:")
    for r in comparison_log:
        if r["diverged"]:
            print(f"  {r['request_id']}: prod='{r['prod_snippet'][:60]}' shadow='{r['shadow_snippet'][:60]}'")

print("\n✓ Shadow deployment demo complete!")

---
## Demo 7: Full Quality Gate Pipeline

All testing layers as a single CI/CD pipeline.
If any layer fails, the pipeline stops and reports why.

This is what runs on every git push in production.

In [ ]:
from anthropic import Anthropic
import json, re, time

client = Anthropic()


def call_model(prompt: str, model: str = "claude-haiku-4-5-20251001", max_tokens: int = 500) -> str:
    resp = client.messages.create(
        model=model, max_tokens=max_tokens, temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text


def run_quality_gate() -> dict:
    results = {"passed": True, "layers": {}}
    t_start = time.time()

    print("\n🚀 Quality Gate Pipeline Starting...")
    print("=" * 55)

    # ─── LAYER 1: Unit Tests ───────────────────────────────
    print("\n[Layer 1] Prompt Unit Tests")
    unit_pass = 0
    unit_tests = [
        ("telnet detection", "Security analysis: line vty 0 4\n transport input telnet\nJSON: {issues: []}", ["telnet"]),
        ("snmp detection", "Security analysis: snmp-server community public RO\nJSON: {issues: []}", ["snmp", "public"]),
        ("json format", "Extract: interface GigabitEthernet0/0\n ip address 10.1.1.1 255.255.255.252\nReturn JSON only: {name, ip}", ["10.1.1.1"]),
    ]
    for name, prompt, keywords in unit_tests:
        resp = call_model(prompt, max_tokens=300)
        found = any(kw.lower() in resp.lower() for kw in keywords)
        icon = "✅" if found else "❌"
        print(f"  {icon} {name}")
        if found: unit_pass += 1
        else: results["passed"] = False

    results["layers"]["unit_tests"] = {"score": unit_pass, "total": len(unit_tests)}

    # ─── LAYER 2: LLM-as-a-Judge ──────────────────────────
    print("\n[Layer 2] LLM-as-a-Judge Quality Check")
    config = """line vty 0 4\n transport input telnet\nsnmp-server community public RO\nno service password-encryption"""
    ai_output = call_model(f"Security analysis:\n{config}\nJSON: {{issues: []}}", max_tokens=600)

    judge_prompt = f"""Score this security analysis (1-5 scale).
Config: {config}
Analysis: {ai_output[:500]}
Ground truth issues: telnet, weak SNMP, password encryption disabled
Return JSON: {{"score": 1-5, "reason": "brief explanation"}}"""

    judge_resp = call_model(judge_prompt, model="claude-sonnet-4-20250514", max_tokens=300)
    m = re.search(r'\{.*?\}', judge_resp, re.DOTALL)
    judge_data = json.loads(m.group()) if m else {"score": 0, "reason": "parse failed"}
    score = judge_data.get("score", 0)
    judge_ok = score >= 4
    icon = "✅" if judge_ok else "❌"
    print(f"  {icon} Quality score: {score}/5 — {judge_data.get('reason', 'N/A')[:80]}")
    if not judge_ok: results["passed"] = False
    results["layers"]["llm_judge"] = {"score": score, "threshold": 4, "passed": judge_ok}

    # ─── LAYER 3: Format/Schema Tests ─────────────────────
    print("\n[Layer 3] Output Schema Tests")
    schema_tests = [
        ("severity field", "Analyze: snmp-server community public RO\nJSON: {severity, issues}", ["severity"]),
        ("array field", "List issues in: no service password-encryption\nJSON: {issues: []}", ["issues", "["]),
    ]
    schema_pass = 0
    for name, prompt, required_patterns in schema_tests:
        resp = call_model(prompt, max_tokens=300)
        ok = all(p.lower() in resp.lower() for p in required_patterns)
        icon = "✅" if ok else "❌"
        print(f"  {icon} {name}")
        if ok: schema_pass += 1
        else: results["passed"] = False
    results["layers"]["schema_tests"] = {"score": schema_pass, "total": len(schema_tests)}

    # ─── FINAL VERDICT ────────────────────────────────────
    elapsed = time.time() - t_start
    print(f"\n{'='*55}")
    print(f"Quality Gate Results ({elapsed:.1f}s):")
    print(f"  Unit Tests : {results['layers']['unit_tests']['score']}/{results['layers']['unit_tests']['total']}")
    print(f"  LLM Judge  : {results['layers']['llm_judge']['score']}/5 (threshold: 4)")
    print(f"  Schema Tests: {results['layers']['schema_tests']['score']}/{results['layers']['schema_tests']['total']}")

    if results["passed"]:
        print("\n✅ QUALITY GATE PASSED — safe to deploy")
    else:
        print("\n❌ QUALITY GATE FAILED — fix issues before deploying")

    return results


# Run it
gate_result = run_quality_gate()
print("\n✓ Full quality gate demo complete!")

---
## Summary

| Testing Layer | What It Catches | Cost | When to Run |
|---------------|-----------------|------|-------------|
| Unit Tests (temp=0) | Prompt regressions, output format | Low (Haiku) | Every commit |
| Regression Baselines | Model drift, behavior changes | Low (Haiku) | Daily |
| LLM-as-a-Judge | Quality, hallucinations, wrong remediation | Medium (Sonnet) | Pre-deploy |
| Semantic Similarity | Synonym failures, meaning drift | Low (embeddings) | Every commit |
| Shadow Mode | Production behavior of new model version | 2x production cost | Before model update |

**The telnet incident would have been caught by**: Layer 1 unit test, Layer 2 baseline regression.
**Cost of prevention**: ~0.01 cents per CI run.
**Cost of the incident**: 1 week of manual re-auditing 400 routers.

**Next**: Chapter 12 — Ethics and Responsible AI. The final chapter of Volume 1.